In [3]:
import pandas as pd
import numpy as np 


In [4]:
df = pd.read_csv("healthcare_noshows_appt.csv")

print(df.shape)
df.head()

(106987, 15)


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,Showed_up,Date.diff
0,2.987250e+13,5642903,F,2016-04-29,2016-04-29,62,JARDIM DA PENHA,False,True,False,False,False,False,True,0
1,5.589978e+14,5642503,M,2016-04-29,2016-04-29,56,JARDIM DA PENHA,False,False,False,False,False,False,True,0
2,4.262962e+12,5642549,F,2016-04-29,2016-04-29,62,MATA DA PRAIA,False,False,False,False,False,False,True,0
3,8.679512e+11,5642828,F,2016-04-29,2016-04-29,8,PONTAL DE CAMBURI,False,False,False,False,False,False,True,0
4,8.841186e+12,5642494,F,2016-04-29,2016-04-29,56,JARDIM DA PENHA,False,True,True,False,False,False,True,0


In [9]:
### Renaming columns

df = df.rename(columns={
    "PatientId": "patient_id",
    "AppointmentID": "appointment_id",
    "ScheduledDay": "scheduled_date",
    "AppointmentDay": "appointment_date",
    "Hipertension" : "hypertension",
    "Handcap" : "handicap",
    "SMS_received": "sms_received",
    "Showed_up": "showed_up",
    "Date.diff": "days_between"
})


In [6]:
### Fixing Data Types

# Convert dates
df["scheduled_date"] = pd.to_datetime(df["scheduled_date"])
df["appointment_date"] = pd.to_datetime(df["appointment_date"])

# Encode target: 1 = No-show, 0 = Show
df["no_show"] = df["showed_up"].map({True: 0, False: 1})

# Encode gender
df["gender"] = df["Gender"].map({"F": 0, "M": 1})


In [7]:
# Missing values
print(df.isnull().sum())

# Remove invalid ages
df = df[df["Age"] >= 0]

# Target distribution
df["no_show"].value_counts(normalize=True)


patient_id          0
appointment_id      0
Gender              0
scheduled_date      0
appointment_date    0
Age                 0
Neighbourhood       0
Scholarship         0
Hipertension        0
Diabetes            0
Alcoholism          0
Handcap             0
sms_received        0
showed_up           0
days_between        0
no_show             0
gender              0
dtype: int64


no_show
0    0.797359
1    0.202641
Name: proportion, dtype: float64

In [10]:
### Feature Engineering

# Day of week (0=Monday)
df["appointment_dayofweek"] = df["appointment_date"].dt.dayofweek
df["is_monday"] = (df["appointment_dayofweek"] == 0).astype(int)

# Previous health conditions flag
df["has_chronic_condition"] = (
    (df["hypertension"] == True) |
    (df["Diabetes"] == True)
).astype(int)


In [11]:
# SMS reminder effect
df.groupby("sms_received")["no_show"].mean()

# No-show rate vs wait time
df.groupby(pd.cut(df["days_between"], bins=[0,1,3,7,14,30,180]))["no_show"].mean()


C:\Users\pthor\AppData\Local\Temp\ipykernel_31824\1660663180.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(pd.cut(df["days_between"], bins=[0,1,3,7,14,30,180]))["no_show"].mean()


days_between
(0, 1]       0.213803
(1, 3]       0.237951
(3, 7]       0.250890
(7, 14]      0.305222
(14, 30]     0.327440
(30, 180]    0.331633
Name: no_show, dtype: float64

In [12]:
df_model = df[[
    "gender",
    "Age",
    "Scholarship",
    "hypertension",
    "Diabetes",
    "Alcoholism",
    "handicap",
    "sms_received",
    "days_between",
    "appointment_dayofweek",
    "is_monday",
    "has_chronic_condition",
    "no_show"
]]

df_model.head()


,gender,Age,Scholarship,hypertension,Diabetes,Alcoholism,handicap,sms_received,days_between,appointment_dayofweek,is_monday,has_chronic_condition,no_show
0,0,62,False,True,False,False,False,False,0,4,0,1,0
1,1,56,False,False,False,False,False,False,0,4,0,0,0
2,0,62,False,False,False,False,False,False,0,4,0,0,0
3,0,8,False,False,False,False,False,False,0,4,0,0,0
4,0,56,False,True,True,False,False,False,0,4,0,1,0


In [13]:
df_model.to_csv("clean_noshows.csv", index=False)
